
# SMaLL-100 (zh->en) — Fine-tuning (Without Wuxia Domain)

**TFG Hugo Silvosa – Baseline NMT (SMaLL-100)**  
Este cuaderno  evalúa un modelo **SMaLL-100** (`alirezamsh/small100`) sin entrenar en dominio usando un dataset de **wuxia** (chino->inglés) ya preparado en formato `datasets` (HF).




## 1) Preparación del entorno

In [ ]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
Nombre de la GPU: NVIDIA GeForce RTX 3060


> **Requisitos del dataset**: directorio HF Datasets con *splits* `train`, `validation`, `test` y columnas `zh` (chino) y `en` (inglés):  
> `processed_data/wuxia_zh_en_clean/`


In [ ]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["evaluation", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Nota: el dataset debe existir en: CORPUS/proccesed data/wuxia_zh_en_clean


Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\trainign
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


## 2) Configuración

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   # <- carpeta con dataset HuggingFace guardado con load_from_disk
    output_dir: Path   = BASE_DIR / "models" / "pretrained_small100_wuxia"              # <- aquí se guardarán los runs/modelos
    ckpt_dir: Path     = BASE_DIR / "checkpoints"                          # <- checkpoints durante entrenamiento
    training_dir: Path = BASE_DIR / "training"         
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    translate_file: Path =   "pre_small100.txt"
    results_file: Path = "pre_results.txt"
    # Columnas del dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Modelo
    src_lang: str = "zh"
    tgt_lang: str = "en"
    model_ckpt: str = "alirezamsh/small100"

    # Entrenamiento
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3

    fraction: float = 1

cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/pretrained_small100_wuxia'), ckpt_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/checkpoints'), training_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/training'), src_col='zh', tgt_col='en', src_lang='zh', tgt_lang='en', model_ckpt='alirezamsh/small100', seed=42, max_source_length=128, max_target_length=128, batch_size=16, epochs=10, learning_rate=2e-05, weight_decay=0.01, early_stopping_patience=3, fraction=1)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas para reproducibilidad
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # Para reproducibilidad estricta
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas y backend configurado.")


Usando dispositivo: cuda
Semillas fijadas y backend configurado.


## 3) Cargar dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"No se encuentra el dataset en: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columnas
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' debe contener columnas '{src_col}' y '{tgt_col}'. Columnas: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

#  pruebas rápidas
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fracción={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})
{'zh': ['第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化', '尤其是看到人群内的宋缺时，神算子立刻警惕，他当年在空城，是第一个跟随白小纯的，受到了重用，如今却成为了第二个，他顿时就视宋缺为竞争对手'], 'en': ['chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun', 'That was even more the case when he noticed Song Que among Bai Xiaochun’s men, which immediately got him even more on guard. Back in Sky City, Master God-Diviner had been the first to start following Bai Xiaochun again, and it had led to incredible benefits. Now, he was only the second to join him, which put Song Que in his sights as a major rival']}
Tam. train/val/test (fracción=1): 417208 52151 52151

## 4) Cargar tokenizador y modelo 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.model_ckpt)


model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_ckpt)
model.to(device)

# Info útil
n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo cargado: {cfg.model_ckpt}")
print(f"Parámetros totales: {n_params:,}")

Modelo cargado: alirezamsh/small100
Parámetros totales: 332,735,488


## 5) Preprocesamiento y tokenización

In [ ]:
# Función para tokenizar ejemplos
def preprocess_function(examples):
    src_list = examples[cfg.src_col]
    tgt_list = examples[cfg.tgt_col]

    cleaned_src, cleaned_tgt = [], []
    for s, t in zip(src_list, tgt_list):
        s = (s or "").strip()
        t = (t or "").strip()
        if s and t:
            cleaned_src.append(s)
            cleaned_tgt.append(t)

    # Idiomas 
    # Para inputs:
    try:
        tokenizer.src_lang = cfg.src_lang
    except Exception:
        pass

    model_inputs = tokenizer(
        cleaned_src,
        max_length=cfg.max_source_length,
        truncation=True,
        padding=False,
    )

    # Para labels: antes de usar text_target, fija tgt_lang
    try:
        tokenizer.tgt_lang = cfg.tgt_lang
    except Exception:
        pass

    labels = tokenizer(
        text_target=cleaned_tgt,
        max_length=cfg.max_target_length,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Aplicar tokenización a todo el dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])

Map:   0%|          | 0/417208 [00:00<?, ? examples/s]

Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

{'input_ids': [128102, 36741, 31659, 22, 18502, 1885, 10195, 2564, 89547, 36664, 29163, 4, 16501, 60116, 44253, 80, 117835, 21082, 4, 91101, 44253, 37256, 51147, 3720, 27321, 32282, 971, 51262, 4, 1957, 8257, 1203, 124909, 5133, 2354, 26528, 1799, 96983, 118223, 4326, 121938, 4, 49121, 2382, 104686, 9600, 116861, 4, 51147, 3720, 5133, 21035, 22105, 80, 26512, 1273, 4, 122102, 11318, 79929, 3453, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [128022, 74070, 436, 7, 1658, 12369, 7, 35701, 1215, 178, 30318, 1776, 241, 4, 1019, 1197, 141, 421, 7, 35701, 1215, 178, 248, 119237, 28, 8, 120052, 4, 122301, 109150, 918, 150, 9792, 124249, 86770, 285, 1307, 84848, 241, 14810, 3972, 128, 4088, 5722, 14810, 9627, 117626, 5091, 4292, 2]}


## 6) Evaluación 

In [ ]:

from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import json
import time

start = time.time()

# Descargar recursos de NLTK (para METEOR)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

#  Parámetros 
EVAL_MAX_SAMPLES = None        # None = todo el split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)

#  Comprobaciones 
assert 'model' in globals(), "No se encontró `model`. Carga el modelo antes."
assert 'tokenizer' in globals(), "No se encontró `tokenizer`. Cárgalo antes."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."

# Asegurar pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id


#  Seleccionar split 
eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No hay ejemplos para evaluar."
def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]


#  Generación por lotes 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
                forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

#  Métricas 
bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

chrf_metric = CHRF(word_order=2)
chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

ter_metric = TER()
ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

def compute_rougeL_f1(hyp_list, ref_list):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    f1s = []
    for h, r in zip(hyp_list, ref_list):
        s = scorer.score(r, h)
        f1s.append(s['rougeL'].fmeasure)
    return float(np.mean(f1s)) * 100.0
rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

def compute_meteor(hyp_list, ref_list):
    scores = []
    for hyp, ref in zip(hyp_list, ref_list):
        hyp_tok = wordpunct_tokenize(hyp) if isinstance(hyp, str) else hyp
        ref_tok = wordpunct_tokenize(ref) if isinstance(ref, str) else ref
        scores.append(meteor_score([ref_tok], hyp_tok))
    return float(np.mean(scores)) * 100.0
meteor_avg = compute_meteor(preds, ref_texts)

end_time = time.time()

results = {
    "model" : cfg.model_ckpt,
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    "sacrebleu": round(bleu_corpus, 4),
    "chrf2": round(chrf_corpus, 4),
    "ter": round(ter_corpus, 4),
    "rougeL_f1": round(rougeL_f1, 4),
    "meteor": round(meteor_avg, 4), 
    "execution_time": round(end_time - start, 2)
}

os.makedirs(cfg.evaluation_dir, exist_ok=True)

res_file = os.path.join(cfg.evaluation_dir, cfg.results_file)

with open(res_file, "a", encoding="utf-8") as f:
    f.write("\n")
    f.write(json.dumps(results, ensure_ascii=False, indent=4))

print(results)



  0%|          | 0/6519 [00:00<?, ?it/s]

{'n_eval': 52151, 'num_beams': 4, 'batch_eval': 8, 'sacrebleu': 0.0002, 'chrf2': 0.309, 'ter': 99.9998, 'rougeL_f1': 0.0113, 'meteor': 2.3417}


## 7) Muestra cualitativa (n ejemplos aleatorios)

In [ ]:
import random
import torch

# Mostrar predicciones aleatorias 
n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decodificar texto fuente y referencia desde dataset tokenizado
    zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # Tokenizar entrada y mover a dispositivo
    inputs = tokenizer(zh, return_tensors="pt").to(device)

    # Generar traducción
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True,
            forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh)
    print("EN (ref):", en_ref)
    print("EN (pred):", en_pred)


ZH: 看着那上面一个个有的熟悉,有的陌生的名字,白小纯的身体越发的颤抖,嘴唇哆嗦着
EN (ref): Some of the names were familiar, others were not, but they all caused him to tremble
EN (pred): ,这就是芬九歌的招标杀案之一,当初在落叶谷中激战,陈百胜一方都吃过这个招呼头。
ZH: 大树地灵告知方源它的认主条件——十万块仙元石,还有大量珍稀的仙材,七转、八转的不计其数,甚至还要求元莲仙尊的鲜血
EN (ref): The tree land spirit told Fang Yuan the ownership condition — a hundred thousand immortal essence stones, large amounts of immortal materials, including many rank seven and rank eight materials, it even wanted the fresh blood of Genesis Lotus Immortal Venerable
EN (pred): ,这就是芬九歌的招标杀案之一,当初在落叶谷中激战,陈百胜一方都吃过这个招呼头。
ZH: 过的还行
EN (ref): It was rather alright
EN (pred): ,这就是芬九歌的招标杀案之一,当初在落叶谷中激战,陈百胜一方都吃过这个招呼头。
ZH: 轮飞嘿嘿一笑,眼中闪烁着阴险的光泽,道
EN (ref): Lun Fei sneered as a sinister light flashed in his eyes
EN (pred): ,这就是芬九歌的招标杀案之一,当初在落叶谷中激战,陈百胜一方都吃过这个招呼头。
ZH: 过了半,欧阳锋走到洪七公身前,说道
EN (ref): After about half a day Ouyang Feng came to Hong Qigong and said
EN (pred): ,这就是芬九歌的招标杀案之一,当初在落叶谷中激战,陈百胜一方都吃过这个招呼头。
ZH: 信来信往,几次之后,方源
EN (ref): After a few times, F

KeyboardInterrupt: 